# RITMO Pipeline - Validación Completa

**Estados Ocultos como Embeddings Estructurados**

Pipeline:
1. RevIN → Normalización reversible
2. Baum-Welch → Entrenamiento HMM (K=5 tokens)
3. Viterbi → Tokenización
4. Embeddings → e_k = [μ_k, σ_k, A[k,:]]

In [ ]:
import os
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')

print(f'Working directory: {Path.cwd()}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from utils.revin import RevINNormalizer
from hmm import baum_welch, viterbi_decode, save_hmm_params, load_hmm_params
from embeddings import EmbeddingGenerator

# Configuración matplotlib (paper-ready)
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

colors_oi = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']

print('Setup completo')

## 1. Carga ETTh1

In [ ]:
df = pd.read_csv('dataset/ETT-small/ETTh1.csv')
data = df['OT'].values

data_train = data[:8640]
data_val = data[8640:8640+2880]
data_test = data[8640+2880:]

print('[ETTh1 DATASET]')
print(f'  Train: {len(data_train)} samples')
print(f'  Val:   {len(data_val)} samples')
print(f'  Test:  {len(data_test)} samples')

## 2. RevIN Normalización

In [ ]:
revin = RevINNormalizer()
normalized_data = revin.fit_transform(data_train)
data_train_norm = normalized_data['train']
stats = revin.get_statistics('train')

success, mse = revin.validate_reconstruction(data_train, data_train_norm, 'train')

print('[REVIN NORMALIZACIÓN]')
print(f'  Pre:  μ={data_train.mean():.3f}, σ={data_train.std():.3f}')
print(f'  Post: μ={data_train_norm.mean():.6f}, σ={data_train_norm.std():.6f}')
print(f'  Reversibilidad: MSE={mse:.2e}, Success={success}')

# VIZ MEJORADA: 3 paneles con estadísticas detalladas
fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(3, 2, height_ratios=[2, 2, 1])

ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :])
ax3 = fig.add_subplot(gs[2, 0])
ax4 = fig.add_subplot(gs[2, 1])

t_viz = 1000

# Panel 1: Original
ax1.plot(data_train[:t_viz], color=colors_oi[0], linewidth=1, alpha=0.8)
ax1.axhline(stats['mean'], color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'μ={stats["mean"]:.2f}')
ax1.fill_between(range(t_viz), stats['mean']-stats['stdev'], stats['mean']+stats['stdev'], 
                alpha=0.15, color='red', label=f'μ±σ (σ={stats["stdev"]:.2f})')
ax1.set_ylabel('Valor Original')
ax1.set_title(f'(A) Serie Original | μ={stats["mean"]:.2f}, σ={stats["stdev"]:.2f}, Range=[{data_train.min():.2f}, {data_train.max():.2f}]', 
             fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(alpha=0.3)

# Panel 2: Normalizada
ax2.plot(data_train_norm[:t_viz], color=colors_oi[4], linewidth=1, alpha=0.8)
ax2.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='μ=0 (target)')
ax2.fill_between(range(t_viz), -1, 1, alpha=0.15, color='gray', label='μ±σ (σ=1 target)')
ax2.set_ylabel('Valor Normalizado')
ax2.set_xlabel('Time')
ax2.set_title(f'(B) Serie Normalizada | μ={data_train_norm.mean():.6f}, σ={data_train_norm.std():.6f}, MSE_recon={mse:.2e}', 
             fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(alpha=0.3)

# Panel 3: Distribución Original
ax3.hist(data_train, bins=50, color=colors_oi[0], alpha=0.7, edgecolor='black', density=True)
x = np.linspace(data_train.min(), data_train.max(), 100)
ax3.plot(x, stats.norm.pdf(x, stats['mean'], stats['stdev']), 'r--', linewidth=2, label='N(μ,σ²)')
ax3.set_xlabel('Valor')
ax3.set_ylabel('Densidad')
ax3.set_title('(C) Distribución Original', fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# Panel 4: Distribución Normalizada
ax4.hist(data_train_norm, bins=50, color=colors_oi[4], alpha=0.7, edgecolor='black', density=True)
x_norm = np.linspace(data_train_norm.min(), data_train_norm.max(), 100)
ax4.plot(x_norm, stats.norm.pdf(x_norm, 0, 1), 'r--', linewidth=2, label='N(0,1) target')
ax4.set_xlabel('Valor')
ax4.set_ylabel('Densidad')
ax4.set_title('(D) Distribución Normalizada', fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fase1_revin.pdf')
plt.savefig('notebooks/fase1_revin.png', dpi=300)
plt.show()
print('\n[GUARDADO] fase1_revin.pdf + .png')

## 3. Baum-Welch Training

In [ ]:
K = 5
cache_path = Path('cache/hmm_etth1_K5.pth')
cache_path.parent.mkdir(exist_ok=True)

print('[BAUM-WELCH TRAINING]')
print(f'  K={K} tokens (vocabulario)')
print(f'  T={len(data_train_norm)} timesteps')

if cache_path.exists():
    print(f'  Cargando HMM desde cache...')
    hmm_params = load_hmm_params(str(cache_path))
    A = hmm_params['A']
    pi = hmm_params['pi']
    mu = hmm_params['mu']
    sigma = hmm_params['sigma']
    log_likelihoods = hmm_params.get('log_likelihoods', [])
    print(f'  ✓ HMM cargado desde cache')
else:
    print(f'  Entrenando HMM desde cero (max_iter=200)...')
    result = baum_welch(
        data_train_norm,
        K=K,
        max_iter=200,
        epsilon=1e-4,
        verbose=True
    )
    
    A = result['A']
    pi = result['pi']
    mu = result['mu']
    sigma = result['sigma']
    log_likelihoods = result.get('log_likelihoods', [])
    
    print(f'\n  Convergió: {result["converged"]}')
    print(f'  Iteraciones: {result["n_iter"]}')
    print(f'  Log-likelihood: {result["log_likelihood"]:.2f}')
    
    save_params = {
        'A': A, 'pi': pi, 'mu': mu, 'sigma': sigma,
        'log_likelihoods': log_likelihoods,
        'K': K
    }
    save_hmm_params(save_params, str(cache_path))
    print(f'  ✓ HMM guardado en {cache_path}')

# VIZ MEJORADA: Convergencia con detalles
if len(log_likelihoods) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # Panel 1: Convergencia completa
    ax1.plot(log_likelihoods, color=colors_oi[4], linewidth=1.5, marker='o', markersize=2, alpha=0.7)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Log-likelihood')
    ax1.set_title(f'(A) Convergencia Baum-Welch | Final LL={log_likelihoods[-1]:.2f}, Iters={len(log_likelihoods)}', 
                 fontweight='bold')
    ax1.grid(alpha=0.3)
    ax1.axhline(log_likelihoods[-1], color='red', linestyle='--', alpha=0.5, label=f'Final: {log_likelihoods[-1]:.2f}')
    ax1.legend()
    
    # Panel 2: Delta LL (tasa de cambio)
    if len(log_likelihoods) > 1:
        deltas = np.diff(log_likelihoods)
        ax2.plot(deltas, color=colors_oi[5], linewidth=1.5, marker='o', markersize=2, alpha=0.7)
        ax2.axhline(1e-4, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Threshold=1e-4')
        ax2.set_xlabel('Iteration')
        ax2.set_ylabel('ΔLL (cambio)')
        ax2.set_title(f'(B) Tasa de Cambio | ΔLL final={deltas[-1]:.2e}', fontweight='bold')
        ax2.set_yscale('log')
        ax2.grid(alpha=0.3)
        ax2.legend()
    
    plt.tight_layout()
    plt.savefig('notebooks/fase2_baum_welch.pdf')
    plt.savefig('notebooks/fase2_baum_welch.png', dpi=300)
    plt.show()
    print('[GUARDADO] fase2_baum_welch.pdf + .png')

## 4. Viterbi Tokenización

In [ ]:
states, log_prob = viterbi_decode(data_train_norm, A, pi, mu, sigma)

state_changes = np.sum(states[1:] != states[:-1])
compression_ratio = len(data_train_norm) / (state_changes + 1)

print('[VITERBI TOKENIZACIÓN]')
print(f'  Vocabulario: K={K} tokens')
print(f'  Secuencia: T={len(states)} tokens (1 por timestep)')
print(f'  Segmentos: {state_changes+1} (run-length)')
print(f'  Ratio compresión: {compression_ratio:.2f}x')
print(f'  Log-probabilidad: {log_prob:.2f}')
print(f'\n  Distribución tokens:')
for k in range(K):
    count = np.sum(states == k)
    pct = 100 * count / len(states)
    print(f'    Token {k}: {count:4d} ({pct:5.2f}%)')

# VIZ MEJORADA: Serie + barra de estados + estadísticas por token
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(4, 2, height_ratios=[2, 0.3, 1.5, 1.5], width_ratios=[2, 1])

ax_series = fig.add_subplot(gs[0, :])
ax_bar = fig.add_subplot(gs[1, :])
ax_hist = fig.add_subplot(gs[2, 0])
ax_duration = fig.add_subplot(gs[2, 1])
ax_params = fig.add_subplot(gs[3, 0])
ax_trans = fig.add_subplot(gs[3, 1])

t_viz = 2000
colors_tokens = plt.cm.tab10(np.linspace(0, 1, K))

# Panel A: Serie con tokens coloreados
ax_series.plot(data_train_norm[:t_viz], color='gray', linewidth=0.6, alpha=0.5, zorder=1)
for k in range(K):
    mask = states[:t_viz] == k
    if np.any(mask):
        ax_series.scatter(np.where(mask)[0], data_train_norm[:t_viz][mask],
                         color=colors_tokens[k], s=8, alpha=0.7, label=f'Token {k} (μ={mu[k]:.2f})', zorder=2)
        ax_series.axhline(mu[k], color=colors_tokens[k], linestyle='--', linewidth=1, alpha=0.4)

ax_series.set_ylabel('Valor Normalizado')
ax_series.set_title(f'(A) Tokenización Viterbi | {len(states)} tokens → {state_changes+1} segmentos (CR={compression_ratio:.1f}x)', 
                   fontweight='bold')
ax_series.legend(loc='upper right', fontsize=7, ncol=3)
ax_series.grid(alpha=0.3)

# Panel B: Barra de estados
for i in range(len(states[:t_viz])):
    ax_bar.axvspan(i, i+1, color=colors_tokens[states[i]], alpha=0.9)
ax_bar.set_xlim(0, t_viz)
ax_bar.set_yticks([])
ax_bar.set_ylabel('Token')
ax_bar.set_title('(B) Secuencia de Tokens', fontweight='bold', fontsize=9)

# Panel C: Distribución de tokens
token_counts = [np.sum(states == k) for k in range(K)]
ax_hist.bar(range(K), token_counts, color=colors_tokens, alpha=0.8, edgecolor='black')
ax_hist.set_xlabel('Token ID')
ax_hist.set_ylabel('Frecuencia')
ax_hist.set_title('(C) Distribución de Tokens', fontweight='bold')
ax_hist.grid(alpha=0.3, axis='y')
for k in range(K):
    pct = 100 * token_counts[k] / len(states)
    ax_hist.text(k, token_counts[k], f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

# Panel D: Duración de segmentos
segment_starts = np.where(np.diff(np.concatenate([[states[0]], states])) != 0)[0]
segment_lengths = np.diff(np.concatenate([segment_starts, [len(states)]]))
ax_duration.hist(segment_lengths, bins=30, color=colors_oi[2], alpha=0.7, edgecolor='black')
ax_duration.axvline(segment_lengths.mean(), color='red', linestyle='--', linewidth=2, label=f'Media={segment_lengths.mean():.1f}')
ax_duration.set_xlabel('Duración (timesteps)')
ax_duration.set_ylabel('Frecuencia')
ax_duration.set_title('(D) Duración de Segmentos', fontweight='bold')
ax_duration.legend()
ax_duration.grid(alpha=0.3, axis='y')

# Panel E: Parámetros por token
x_pos = np.arange(K)
ax_params.bar(x_pos - 0.2, mu, 0.4, label='μ (mean)', color=colors_oi[4], alpha=0.8)
ax_params.bar(x_pos + 0.2, sigma, 0.4, label='σ (std)', color=colors_oi[5], alpha=0.8)
ax_params.set_xlabel('Token ID')
ax_params.set_ylabel('Valor')
ax_params.set_title('(E) Parámetros HMM por Token', fontweight='bold')
ax_params.set_xticks(x_pos)
ax_params.legend()
ax_params.grid(alpha=0.3, axis='y')
ax_params.axhline(0, color='black', linewidth=0.8)

# Panel F: Matriz de transición
im = ax_trans.imshow(A, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax_trans.set_xlabel('To Token j')
ax_trans.set_ylabel('From Token i')
ax_trans.set_title('(F) Matriz Transición A', fontweight='bold')
ax_trans.set_xticks(range(K))
ax_trans.set_yticks(range(K))
for i in range(K):
    for j in range(K):
        ax_trans.text(j, i, f'{A[i,j]:.2f}', ha='center', va='center',
                     color='white' if A[i,j] > 0.5 else 'black', fontsize=8)
plt.colorbar(im, ax=ax_trans, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('notebooks/fase3_viterbi.pdf')
plt.savefig('notebooks/fase3_viterbi.png', dpi=300)
plt.show()
print('\n[GUARDADO] fase3_viterbi.pdf + .png')

## 5. Embeddings Estructurados

In [ ]:
hmm_params_dict = {'A': A, 'mu': mu, 'sigma': sigma, 'pi': pi}
emb_gen = EmbeddingGenerator(hmm_params_dict, d_model=128, device='cpu')
embedding_table = emb_gen.get_embedding_table().numpy()

print('[EMBEDDINGS ESTRUCTURADOS]')
print(f'  e_k = [μ_k, σ_k, A[k,:]]')
print(f'  K={K} tokens → {K} embeddings')
print(f'  Dimensión cruda: {embedding_table.shape[1]} = 2 + K')
print(f'  d_model (proyectado): {emb_gen.d_model}')
print(f'\n  Ejemplo token 0:')
print(f'    μ_0={mu[0]:.3f}, σ_0={sigma[0]:.3f}')
print(f'    A[0,:]=[{" ".join(f"{x:.3f}" for x in A[0,:])}]')

# VIZ MEJORADA: Múltiples vistas del espacio de embeddings
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3)

ax1 = fig.add_subplot(gs[0, :2])
ax2 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[1, 2])
ax6 = fig.add_subplot(gs[2, :])

# Panel A: μ vs σ con anotaciones detalladas
for k in range(K):
    ax1.scatter(mu[k], sigma[k], c=[colors_tokens[k]], s=300, edgecolors='black', 
               linewidths=2, zorder=3, alpha=0.9)
    ax1.annotate(f'T{k}\nμ={mu[k]:.2f}\nσ={sigma[k]:.2f}', 
                (mu[k], sigma[k]), fontsize=8, ha='center', va='center',
                color='white', fontweight='bold', zorder=4)
    
    # Dibujar círculo de radio σ
    circle = plt.Circle((mu[k], 0), sigma[k], fill=False, color=colors_tokens[k], 
                       linestyle='--', alpha=0.3, linewidth=1.5)
    ax1.add_patch(circle)

ax1.set_xlabel('μ_k (regime mean)')
ax1.set_ylabel('σ_k (volatility)')
ax1.set_title('(A) Espacio μ-σ | Cada token captura un régimen estadístico', fontweight='bold')
ax1.grid(alpha=0.3)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.axvline(0, color='black', linewidth=0.8)

# Panel B: Matriz A con heatmap
im = ax2.imshow(A, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax2.set_xlabel('To Token')
ax2.set_ylabel('From Token')
ax2.set_title('(B) Matriz A', fontweight='bold')
ax2.set_xticks(range(K))
ax2.set_yticks(range(K))
for i in range(K):
    for j in range(K):
        ax2.text(j, i, f'{A[i,j]:.2f}', ha='center', va='center',
                color='white' if A[i,j] > 0.5 else 'black', fontsize=8)
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

# Panel C: Distribución π inicial
ax3.bar(range(K), pi, color=colors_tokens, alpha=0.8, edgecolor='black')
ax3.set_xlabel('Token ID')
ax3.set_ylabel('π (prob inicial)')
ax3.set_title('(C) Distribución Inicial π', fontweight='bold')
ax3.grid(alpha=0.3, axis='y')

# Panel D: Auto-transiciones (diagonal de A)
self_trans = np.diag(A)
ax4.bar(range(K), self_trans, color=colors_tokens, alpha=0.8, edgecolor='black')
ax4.set_xlabel('Token ID')
ax4.set_ylabel('A[k,k] (persistencia)')
ax4.set_title('(D) Persistencia de Tokens', fontweight='bold')
ax4.grid(alpha=0.3, axis='y')
for k in range(K):
    ax4.text(k, self_trans[k], f'{self_trans[k]:.2f}', ha='center', va='bottom', fontsize=8)

# Panel E: Embedding crudo (primeros 7 dims)
ax5.imshow(embedding_table, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
ax5.set_xlabel('Dimensión')
ax5.set_ylabel('Token ID')
ax5.set_title('(E) Embeddings Crudos [μ,σ,A]', fontweight='bold')
ax5.set_yticks(range(K))
ax5.set_xticks(range(embedding_table.shape[1]))
ax5.set_xticklabels(['μ', 'σ'] + [f'A{i}' for i in range(K)], fontsize=7)

# Panel F: Composición del embedding (desglose)
for k in range(K):
    emb = embedding_table[k]
    ax6.barh(k*3, emb[0], color=colors_oi[4], alpha=0.8, label='μ' if k==0 else '')
    ax6.barh(k*3+0.8, emb[1], color=colors_oi[5], alpha=0.8, label='σ' if k==0 else '')
    ax6.text(emb[0], k*3, f'{emb[0]:.2f}', va='center', fontsize=7)
    ax6.text(emb[1], k*3+0.8, f'{emb[1]:.2f}', va='center', fontsize=7)

ax6.set_yticks([k*3+0.4 for k in range(K)])
ax6.set_yticklabels([f'Token {k}' for k in range(K)])
ax6.set_xlabel('Valor')
ax6.set_title('(F) Composición Embeddings: μ (azul) y σ (naranja)', fontweight='bold')
ax6.legend(loc='upper right')
ax6.grid(alpha=0.3, axis='x')
ax6.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('notebooks/fase4_embeddings.pdf')
plt.savefig('notebooks/fase4_embeddings.png', dpi=300)
plt.show()
print('\n[GUARDADO] fase4_embeddings.pdf + .png')

## Conclusiones

Pipeline RITMO validado:

1. ✅ RevIN: Normalización reversible (MSE~0)
2. ✅ Baum-Welch: HMM con K=5 tokens converge
3. ✅ Viterbi: Secuencia de T tokens → compresión ~27x
4. ✅ Embeddings: e_k = [μ_k, σ_k, A[k,:]]

**Próximo paso:** Integración Transformer (models/RITMO.py)